<a href="https://colab.research.google.com/github/chetankumar2589/Enhanced_SkinCancer_Classification/blob/main/Enhanced_SkinCancer_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***PHASE 1: Kaggle Setup & HAM10000 Download in Colab***

In [ ]:
# Install Kaggle package
!pip install -q kaggle

In [ ]:
from google.colab import files

# Upload kaggle.json
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Move to correct location
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle credentials configured!")

In [ ]:
# Download dataset
print("Downloading HAM10000 dataset... (This may take 2-3 minutes)")
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

print("✅ Download complete!")

In [ ]:
import zipfile
import os

# Unzip the dataset
print("Unzipping dataset...")
with zipfile.ZipFile('skin-cancer-mnist-ham10000.zip', 'r') as zip_ref:
    zip_ref.extractall('HAM10000')

print("✅ Dataset extracted to 'HAM10000' folder")

# Check contents
print("\nDataset contents:")
!ls -lh HAM10000/

In [ ]:
import pandas as pd

# Load metadata
metadata = pd.read_csv('HAM10000/HAM10000_metadata.csv')

print(f"Total images: {len(metadata)}")
print(f"\nClass distribution:")
print(metadata['dx'].value_counts())

print("\n✅ PHASE 1 COMPLETE!")
print("Dataset ready. Tell me: 'Phase 1 done, move to Phase 2'")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

***PHASE 2: Data Preprocessing & Imbalance Handling***

In [ ]:
# Install necessary libraries
!pip install -q albumentations timm torch torchvision

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Libraries imported successfully!")

In [ ]:
# Label mapping (as per CRITICAL REQUIREMENT)
lesion_type_dict = {
    'nv': 'Melanocytic nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis-like lesions',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}

# Numeric mapping for model training
label_to_idx = {label: idx for idx, label in enumerate(lesion_type_dict.keys())}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print("Label Mapping:")
for abbr, full_name in lesion_type_dict.items():
    print(f"  {abbr} ({label_to_idx[abbr]}) → {full_name}")

In [ ]:
# Load metadata
metadata = pd.read_csv('HAM10000/HAM10000_metadata.csv')

# Find all image files (HAM10000 splits images into 2 folders)
image_folder_1 = 'HAM10000/HAM10000_images_part_1'
image_folder_2 = 'HAM10000/HAM10000_images_part_2'

# Create full image paths
def get_image_path(image_id):
    path1 = os.path.join(image_folder_1, f"{image_id}.jpg")
    path2 = os.path.join(image_folder_2, f"{image_id}.jpg")
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None

metadata['image_path'] = metadata['image_id'].apply(get_image_path)

# Remove any rows with missing images
metadata = metadata[metadata['image_path'].notna()].reset_index(drop=True)

# Add numeric labels
metadata['label'] = metadata['dx'].map(label_to_idx)

print(f"✅ Total valid images: {len(metadata)}")
print(f"\nClass distribution:")
print(metadata['dx'].value_counts())

In [ ]:
# Calculate class weights for WeightedRandomSampler
class_counts = metadata['dx'].value_counts().to_dict()
total_samples = len(metadata)

# Compute weights (inverse frequency)
class_weights = {label: total_samples / count for label, count in class_counts.items()}

# Map to each sample
sample_weights = metadata['dx'].map(class_weights).values

print("Class Weights (for oversampling minority classes):")
for label, weight in class_weights.items():
    print(f"  {lesion_type_dict[label]}: {weight:.2f}")

print("\n✅ Class imbalance handling prepared!")

In [ ]:
# Split: 70% train, 20% val, 10% test
train_df, temp_df = train_test_split(
    metadata,
    test_size=0.3,
    stratify=metadata['dx'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.33,  # 0.33 of 30% = 10% of total
    stratify=temp_df['dx'],
    random_state=42
)

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

print("\n✅ PHASE 2 COMPLETE!")
print("Data preprocessing done. Tell me: 'Phase 2 done, move to Phase 3'")

***PHASE 3: Dataset Class & Data Augmentation***

In [ ]:
# Training augmentations (224x224 for now)
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.CLAHE(p=0.3),  # Contrast enhancement
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),  # Cutout
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Validation/Test transforms (no augmentation)
val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("✅ Augmentation pipelines created!")

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Load image
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Get label
        label = self.df.loc[idx, 'label']

        # Apply augmentations
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label

print("✅ Dataset class defined!")

In [ ]:
# Create datasets
train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset = HAM10000Dataset(val_df, transform=val_transform)
test_dataset = HAM10000Dataset(test_df, transform=val_transform)

# WeightedRandomSampler for training (handles imbalance)
train_weights = train_df['dx'].map(class_weights).values
train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)

# Create DataLoaders
batch_size = 32  # Adjust based on GPU memory

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,  # Uses weighted sampling
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Visualize some augmented training samples
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Get a batch
images, labels = next(iter(train_loader))

# Plot 8 samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    label_abbr = idx_to_label[labels[i].item()]
    label_full = lesion_type_dict[label_abbr]

    ax.imshow(img)
    ax.set_title(f"{label_full}\n({label_abbr})", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✅ PHASE 3 COMPLETE!")
print("Dataset & DataLoaders ready. Tell me: 'Phase 3 done, move to Phase 4'")

***PHASE 4: Teacher Model - Backbone A (ConvNeXt V2)***

In [ ]:
import timm

# Check available ConvNeXt V2 models
print("Available ConvNeXt V2 models:")
convnext_models = [m for m in timm.list_models() if 'convnextv2' in m.lower()]
for model in convnext_models[:10]:
    print(f"  - {model}")

# Load ConvNeXt V2 Base (pretrained)
print("\nLoading ConvNeXt V2 Base...")
convnext_backbone = timm.create_model(
    'convnextv2_base.fcmae_ft_in22k_in1k',  # Pretrained with MAE
    pretrained=True,
    features_only=True,  # Extract feature maps
    out_indices=[2, 3]   # Last 2 stages (stage 3, stage 4)
)

# Set to eval mode for now
convnext_backbone.eval()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
convnext_backbone = convnext_backbone.to(device)

print(f"✅ ConvNeXt V2 loaded on {device}")

In [ ]:
# Create dummy input (batch_size=2, channels=3, height=224, width=224)
dummy_input = torch.randn(2, 3, 224, 224).to(device)

# Extract features
with torch.no_grad():
    convnext_features = convnext_backbone(dummy_input)

print("ConvNeXt V2 Feature Maps:")
for i, feat in enumerate(convnext_features):
    print(f"  Stage {i+3}: {feat.shape}")  # Stage 3, Stage 4

print("\n✅ Feature extraction working!")

In [ ]:
# First, let's check how many stages EfficientNet V2 has
print("Checking EfficientNet V2 architecture...")
temp_model = timm.create_model(
    'tf_efficientnetv2_m.in21k_ft_in1k',
    pretrained=False,
    features_only=True
)

# Get feature info
print(f"Total stages available: {len(temp_model.feature_info)}")
for i, info in enumerate(temp_model.feature_info):
    print(f"  Stage {i}: channels={info['num_chs']}, reduction={info['reduction']}")

del temp_model  # Free memory

In [ ]:
# Load EfficientNet V2 Medium (pretrained)
print("\nLoading EfficientNet V2 Medium...")
efficientnet_backbone = timm.create_model(
    'tf_efficientnetv2_m.in21k_ft_in1k',  # Pretrained on ImageNet-21k
    pretrained=True,
    features_only=True,  # Extract feature maps
    out_indices=[3, 4]   # FIXED: Use last 2 available stages (usually 0-4, so 3,4)
)

efficientnet_backbone.eval()
efficientnet_backbone = efficientnet_backbone.to(device)

print(f"✅ EfficientNet V2 loaded on {device}")

In [ ]:
# Test with dummy input
dummy_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    efficientnet_features = efficientnet_backbone(dummy_input)

print("EfficientNet V2 Feature Maps:")
for i, feat in enumerate(efficientnet_features):
    print(f"  Stage {i}: {feat.shape}")

print("\n✅ EfficientNet V2 feature extraction working!")

In [ ]:
# Test with actual batch size
test_input = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    conv_feats = convnext_backbone(test_input)
    eff_feats = efficientnet_backbone(test_input)

print("\n📊 Feature Map Shapes (for 224x224 input):")
print("\nConvNeXt V2:")
for i, f in enumerate(conv_feats):
    print(f"  Stage {i}: {f.shape} (C={f.shape[1]}, H={f.shape[2]}, W={f.shape[3]})")

print("\nEfficientNet V2:")
for i, f in enumerate(eff_feats):
    print(f"  Stage {i}: {f.shape} (C={f.shape[1]}, H={f.shape[2]}, W={f.shape[3]})")

# Store for MSAF module
all_features = conv_feats + eff_feats
print(f"\n✅ Total feature maps to fuse: {len(all_features)}")

print("\n✅ PHASE 4 COMPLETE!")
print("Both backbones working. Tell me: 'Phase 4 done, move to Phase 5'")

# ***PHASE 5: MSAF (Multi-Stage Attention Fusion) Module***

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x

print("✅ CBAM Attention module defined!")

In [ ]:
class MSAF(nn.Module):
    """Multi-Stage Attention Fusion - fuses features from multiple backbones"""
    def __init__(self, feature_channels, target_size=(14, 14), output_channels=512):
        """
        Args:
            feature_channels: List of channel counts for each feature map
            target_size: Target spatial size (H, W) to upsample to
            output_channels: Final output channel count after fusion
        """
        super().__init__()
        self.target_size = target_size

        # Create CBAM for each feature map
        self.attention_modules = nn.ModuleList([
            CBAM(channels) for channels in feature_channels
        ])

        # Calculate total channels after concatenation
        total_channels = sum(feature_channels)

        # 1x1 Conv to reduce channels after fusion
        self.channel_reduction = nn.Sequential(
            nn.Conv2d(total_channels, output_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, feature_list):
        """
        Args:
            feature_list: List of feature maps from different backbones
        Returns:
            Fused feature map
        """
        refined_features = []

        # Apply attention to each feature map
        for feat, attn_module in zip(feature_list, self.attention_modules):
            refined = attn_module(feat)

            # Upsample to target size if needed
            if refined.shape[2:] != self.target_size:
                refined = nn.functional.interpolate(
                    refined,
                    size=self.target_size,
                    mode='bilinear',
                    align_corners=False
                )

            refined_features.append(refined)

        # Concatenate along channel dimension
        fused = torch.cat(refined_features, dim=1)

        # Reduce channels
        output = self.channel_reduction(fused)

        return output

print("✅ MSAF module defined!")

In [ ]:
# Get feature shapes from our backbones
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    conv_feats = convnext_backbone(test_input)
    eff_feats = efficientnet_backbone(test_input)

# Combine features
all_feats = conv_feats + eff_feats

# Get channel counts
feature_channels = [f.shape[1] for f in all_feats]
print(f"Feature channels: {feature_channels}")

# Find largest spatial size (will be our target)
spatial_sizes = [(f.shape[2], f.shape[3]) for f in all_feats]
target_size = max(spatial_sizes, key=lambda x: x[0] * x[1])
print(f"Target spatial size: {target_size}")

In [ ]:
# Create MSAF module
msaf = MSAF(
    feature_channels=feature_channels,
    target_size=target_size,
    output_channels=512  # Reduced dimension for Transformer
).to(device)

print(f"✅ MSAF initialized!")
print(f"   Input features: {len(feature_channels)} maps")
print(f"   Total input channels: {sum(feature_channels)}")
print(f"   Output channels: 512")

In [ ]:
# Test forward pass
with torch.no_grad():
    fused_features = msaf(all_feats)

print(f"\n📊 MSAF Output Shape: {fused_features.shape}")
print(f"   Batch: {fused_features.shape[0]}")
print(f"   Channels: {fused_features.shape[1]}")
print(f"   Spatial: {fused_features.shape[2]}x{fused_features.shape[3]}")

print("\n✅ PHASE 5 COMPLETE!")
print("MSAF module working. Tell me: 'Phase 5 done, move to Phase 6'")

# ***PHASE 6: Transformer Refinement Head***

In [ ]:
class TransformerRefinementHead(nn.Module):
    """Lightweight Transformer to capture global context after fusion"""
    def __init__(self, feature_channels=512, num_heads=4, num_layers=2, dim_feedforward=512):
        """
        Args:
            feature_channels: Input channel dimension (from MSAF output)
            num_heads: Number of attention heads
            num_layers: Number of Transformer encoder layers
            dim_feedforward: Hidden dimension in FFN
        """
        super().__init__()

        # Adaptive pooling to get fixed spatial size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))  # 7x7 = 49 tokens

        # Positional embedding (learnable)
        self.pos_embedding = nn.Parameter(torch.randn(1, 49, feature_channels))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_channels,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Global average pooling after transformer
        self.gap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        """
        Args:
            x: Feature map from MSAF [B, C, H, W]
        Returns:
            Global feature vector [B, C]
        """
        B, C, H, W = x.shape

        # Adaptive pool to fixed size
        x = self.adaptive_pool(x)  # [B, C, 7, 7]

        # Reshape to sequence: [B, C, 7, 7] -> [B, 49, C]
        x = x.flatten(2).permute(0, 2, 1)  # [B, 49, C]

        # Add positional embedding
        x = x + self.pos_embedding

        # Pass through Transformer
        x = self.transformer_encoder(x)  # [B, 49, C]

        # Global average pooling: [B, 49, C] -> [B, C, 49] -> [B, C, 1] -> [B, C]
        x = x.permute(0, 2, 1)  # [B, C, 49]
        x = self.gap(x).squeeze(-1)  # [B, C]

        return x

print("✅ Transformer Refinement Head defined!")

In [ ]:
# Create Transformer Head
transformer_head = TransformerRefinementHead(
    feature_channels=512,  # Match MSAF output
    num_heads=4,
    num_layers=2,
    dim_feedforward=512
).to(device)

print(f"✅ Transformer Head initialized!")
print(f"   Input: [B, 512, H, W]")
print(f"   Output: [B, 512]")

In [ ]:
# Test with MSAF output
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    # Extract features from backbones
    conv_feats = convnext_backbone(test_input)
    eff_feats = efficientnet_backbone(test_input)
    all_feats = conv_feats + eff_feats

    # Fuse with MSAF
    fused = msaf(all_feats)
    print(f"MSAF output: {fused.shape}")

    # Pass through Transformer
    global_features = transformer_head(fused)
    print(f"Transformer output: {global_features.shape}")

print("\n✅ Transformer Head working!")

In [ ]:
class TeacherModel(nn.Module):
    """Complete Teacher Model: ConvNeXt V2 + EfficientNet V2 + MSAF + Transformer"""
    def __init__(self, num_classes=7):
        super().__init__()

        # Backbone A: ConvNeXt V2
        self.convnext = timm.create_model(
            'convnextv2_base.fcmae_ft_in22k_in1k',
            pretrained=True,
            features_only=True,
            out_indices=[2, 3]
        )

        # Backbone B: EfficientNet V2
        self.efficientnet = timm.create_model(
            'tf_efficientnetv2_m.in21k_ft_in1k',
            pretrained=True,
            features_only=True,
            out_indices=[3, 4]
        )

        # Get feature channels dynamically
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            conv_f = self.convnext(dummy)
            eff_f = self.efficientnet(dummy)
            all_f = conv_f + eff_f
            feature_channels = [f.shape[1] for f in all_f]
            target_size = max([(f.shape[2], f.shape[3]) for f in all_f], key=lambda x: x[0]*x[1])

        # MSAF Module
        self.msaf = MSAF(
            feature_channels=feature_channels,
            target_size=target_size,
            output_channels=512
        )

        # Transformer Head
        self.transformer_head = TransformerRefinementHead(
            feature_channels=512,
            num_heads=4,
            num_layers=2,
            dim_feedforward=512
        )

        # Classification Head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Extract features from both backbones
        conv_feats = self.convnext(x)
        eff_feats = self.efficientnet(x)

        # Combine features
        all_feats = conv_feats + eff_feats

        # Fuse with attention
        fused = self.msaf(all_feats)

        # Global context with Transformer
        global_feat = self.transformer_head(fused)

        # Classification
        logits = self.classifier(global_feat)

        return logits

print("✅ Complete Teacher Model defined!")

In [ ]:
# Create Teacher Model
teacher_model = TeacherModel(num_classes=7).to(device)

# Count parameters
total_params = sum(p.numel() for p in teacher_model.parameters())
trainable_params = sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)

print(f"\n📊 Teacher Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1e6:.1f} MB (fp32)")

# Test forward pass
test_batch = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = teacher_model(test_batch)

print(f"\n✅ Teacher Model forward pass successful!")
print(f"   Input: {test_batch.shape}")
print(f"   Output: {output.shape}")
print(f"   Output range: [{output.min().item():.2f}, {output.max().item():.2f}]")

print("\n✅ PHASE 6 COMPLETE!")
print("Teacher Model ready. Tell me: 'Phase 6 done, move to Phase 7'")

# ***PHASE 7: Training Setup & Teacher Model Training***

In [ ]:
# Training configuration
config = {
    'epochs_224': 25,      # Epochs at 224x224
    'epochs_384': 10,      # Epochs at 384x384 (progressive resizing)
    'batch_size': 32,
    'learning_rate': 3e-4,
    'weight_decay': 0.05,
    'num_classes': 7,
    'device': device
}

print("✅ Training configuration set!")
print(f"   Device: {config['device']}")
print(f"   Phase 1: {config['epochs_224']} epochs @ 224px")
print(f"   Phase 2: {config['epochs_384']} epochs @ 384px")

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

# Loss function (Cross Entropy)
criterion = nn.CrossEntropyLoss()

# Optimizer (AdamW)
optimizer = optim.AdamW(
    teacher_model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Learning rate scheduler (OneCycleLR for 224px phase)
steps_per_epoch = len(train_loader)
total_steps_224 = steps_per_epoch * config['epochs_224']

scheduler = OneCycleLR(
    optimizer,
    max_lr=config['learning_rate'],
    total_steps=total_steps_224,
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=10000.0
)

print("✅ Loss, optimizer, and scheduler initialized!")
print(f"   Total training steps (224px): {total_steps_224}")

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# Initialize GradScaler for mixed precision
scaler = GradScaler()

def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Backward pass with scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update scheduler
        scheduler.step()

        # Calculate accuracy
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        running_loss += loss.item()

        # Print progress every 50 batches
        if (batch_idx + 1) % 50 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            accuracy = 100. * correct / total
            print(f'  Batch [{batch_idx+1}/{len(loader)}] | Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}%')

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc

print("✅ Training function defined!")

In [ ]:
def validate(model, loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Calculate accuracy
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            running_loss += loss.item()

    val_loss = running_loss / len(loader)
    val_acc = 100. * correct / total

    return val_loss, val_acc

print("✅ Validation function defined!")

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}


test_epochs = 20

print(f"\n🚀 Starting Teacher Model Training (224px, {test_epochs} epochs)...")
print("=" * 60)

best_val_acc = 0.0

for epoch in range(test_epochs):
    print(f"\nEpoch [{epoch+1}/{test_epochs}]")
    print("-" * 60)

    # Train
    train_loss, train_acc = train_epoch(
        teacher_model, train_loader, criterion, optimizer, scheduler, scaler, device
    )

    # Validate
    val_loss, val_acc = validate(teacher_model, val_loader, criterion, device)

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Print epoch summary
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(teacher_model.state_dict(), 'teacher_best_224.pth')
        print(f"   ✅ Best model saved! (Val Acc: {best_val_acc:.2f}%)")

print("\n" + "=" * 60)
print(f"✅ Training Complete!")
print(f"   Best Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Acc', marker='o')
ax2.plot(history['val_acc'], label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ PHASE 7 COMPLETE!")
print("Teacher model trained (5 epochs @ 224px). Tell me: 'Phase 7 done, move to Phase 8'")

# ***PHASE 8: Student Model & Knowledge Distillation***

In [ ]:
class StudentModel(nn.Module):
    """Lightweight Student Model for deployment (~3.7M params)"""
    def __init__(self, num_classes=7):
        super().__init__()

        # ConvNeXt V2 Atto (very small, fast model)
        self.backbone = timm.create_model(
            'convnextv2_atto.fcmae_ft_in1k',  # Atto variant
            pretrained=True,
            num_classes=0  # Remove classification head
        )

        # Get feature dimension
        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            features = self.backbone(dummy)
            feature_dim = features.shape[1]

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(feature_dim, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

print("✅ Student Model defined!")

In [ ]:
# Create Student Model
student_model = StudentModel(num_classes=7).to(device)

# Count parameters
student_params = sum(p.numel() for p in student_model.parameters())
teacher_params = sum(p.numel() for p in teacher_model.parameters())

print(f"\n📊 Model Comparison:")
print(f"   Teacher: {teacher_params:,} params (~{teacher_params*4/1e6:.1f} MB)")
print(f"   Student: {student_params:,} params (~{student_params*4/1e6:.1f} MB)")
print(f"   Compression ratio: {teacher_params/student_params:.1f}x smaller")

# Test forward pass
test_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    student_output = student_model(test_input)

print(f"\n✅ Student Model initialized!")
print(f"   Output shape: {student_output.shape}")

In [ ]:
class DistillationLoss(nn.Module):
    """Knowledge Distillation Loss: KL Divergence + Cross Entropy"""
    def __init__(self, temperature=4.0, alpha=0.7):
        """
        Args:
            temperature: Softening parameter for teacher logits
            alpha: Weight for distillation loss (1-alpha for hard label loss)
        """
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.kl_div = nn.KLDivLoss(reduction='batchmean')
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, labels):
        """
        Args:
            student_logits: Raw logits from student [B, num_classes]
            teacher_logits: Raw logits from teacher [B, num_classes]
            labels: Ground truth labels [B]
        Returns:
            Combined loss
        """
        # Soft targets from teacher (with temperature)
        soft_teacher = nn.functional.softmax(teacher_logits / self.temperature, dim=1)
        soft_student = nn.functional.log_softmax(student_logits / self.temperature, dim=1)

        # Distillation loss (KL divergence)
        distillation_loss = self.kl_div(soft_student, soft_teacher) * (self.temperature ** 2)

        # Hard label loss (standard cross entropy)
        hard_loss = self.ce_loss(student_logits, labels)

        # Combined loss
        total_loss = self.alpha * distillation_loss + (1 - self.alpha) * hard_loss

        return total_loss, distillation_loss, hard_loss

print("✅ Distillation Loss defined!")

In [ ]:
# Load best teacher model
teacher_model.load_state_dict(torch.load('teacher_best_224.pth'))
teacher_model.eval()  # Freeze teacher

# Freeze teacher parameters
for param in teacher_model.parameters():
    param.requires_grad = False

print("✅ Teacher model frozen and loaded!")

# Student optimizer
student_optimizer = optim.AdamW(
    student_model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

# Scheduler for student
student_steps = len(train_loader) * 35  # 35 epochs for student
student_scheduler = OneCycleLR(
    student_optimizer,
    max_lr=3e-4,
    total_steps=student_steps,
    pct_start=0.3,
    anneal_strategy='cos'
)

# Distillation loss (T=4, alpha=0.7)
distill_criterion = DistillationLoss(temperature=4.0, alpha=0.7)

print("✅ Student training setup complete!")
print(f"   Training for 20 epochs")
print(f"   Temperature: 4.0")
print(f"   Alpha: 0.7")

In [ ]:
def train_student_epoch(student, teacher, loader, criterion, optimizer, scheduler, scaler, device):
    """Train student with knowledge distillation"""
    student.train()
    teacher.eval()  # Teacher always in eval mode

    running_loss = 0.0
    running_distill_loss = 0.0
    running_hard_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward
        with autocast():
            # Student forward
            student_logits = student(images)

            # Teacher forward (no gradients)
            with torch.no_grad():
                teacher_logits = teacher(images)

            # Distillation loss
            loss, distill_loss, hard_loss = criterion(student_logits, teacher_logits, labels)

        # Backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Metrics
        _, predicted = student_logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        running_loss += loss.item()
        running_distill_loss += distill_loss.item()
        running_hard_loss += hard_loss.item()

        # Progress
        if (batch_idx + 1) % 50 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            accuracy = 100. * correct / total
            print(f'  Batch [{batch_idx+1}/{len(loader)}] | Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}%')

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc

print("✅ Student training function defined!")

In [ ]:
# Student training history
student_history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# Train for 10 epochs (instead of 20 to save time)
test_student_epochs = 30

print(f"\n🚀 Starting Student Training with Distillation ({test_student_epochs} epochs)...")
print("=" * 60)

best_student_acc = 0.0

for epoch in range(test_student_epochs):
    print(f"\nEpoch [{epoch+1}/{test_student_epochs}]")
    print("-" * 60)

    # Train with distillation
    train_loss, train_acc = train_student_epoch(
        student_model, teacher_model, train_loader,
        distill_criterion, student_optimizer, student_scheduler, scaler, device
    )

    # Validate
    val_loss, val_acc = validate(student_model, val_loader, criterion, device)

    # Save history
    student_history['train_loss'].append(train_loss)
    student_history['train_acc'].append(train_acc)
    student_history['val_loss'].append(val_loss)
    student_history['val_acc'].append(val_acc)

    # Print summary
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save best
    if val_acc > best_student_acc:
        best_student_acc = val_acc
        torch.save(student_model.state_dict(), 'student_best.pth')
        print(f"   ✅ Best student saved! (Val Acc: {best_student_acc:.2f}%)")

print("\n" + "=" * 60)
print(f"✅ Student Training Complete!")
print(f"   Best Student Val Acc: {best_student_acc:.2f}%")

In [ ]:
# Load best models
teacher_model.load_state_dict(torch.load('teacher_best_224.pth'))
student_model.load_state_dict(torch.load('student_best.pth'))

# Evaluate on test set
teacher_test_loss, teacher_test_acc = validate(teacher_model, test_loader, criterion, device)
student_test_loss, student_test_acc = validate(student_model, test_loader, criterion, device)

print("\n📊 Final Test Set Performance:")
print("=" * 60)
print(f"Teacher Model:")
print(f"  Test Loss: {teacher_test_loss:.4f}")
print(f"  Test Accuracy: {teacher_test_acc:.2f}%")
print(f"\nStudent Model:")
print(f"  Test Loss: {student_test_loss:.4f}")
print(f"  Test Accuracy: {student_test_acc:.2f}%")
print(f"\nAccuracy Gap: {teacher_test_acc - student_test_acc:.2f}%")
print(f"Model Size Reduction: {teacher_params/student_params:.1f}x")

print("\n✅ PHASE 8 COMPLETE!")
print("Student trained via distillation. Tell me: 'Phase 8 done, move to Phase 9'")

# ***PHASE 9: Evaluation Metrics & Grad-CAM Visualization***

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, f1_score, precision_score, recall_score
)
import seaborn as sns

print("✅ Evaluation libraries imported!")

In [ ]:
def get_predictions(model, loader, device):
    """Get all predictions and labels"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# Get predictions for both models
print("Generating predictions...")
teacher_preds, true_labels, teacher_probs = get_predictions(teacher_model, test_loader, device)
student_preds, _, student_probs = get_predictions(student_model, test_loader, device)

print(f"✅ Predictions generated!")
print(f"   Test samples: {len(true_labels)}")

In [ ]:
# Class names
class_names = list(lesion_type_dict.keys())
class_full_names = list(lesion_type_dict.values())

# Calculate metrics for Student Model
student_accuracy = (student_preds == true_labels).mean() * 100
student_f1_macro = f1_score(true_labels, student_preds, average='macro') * 100
student_precision = precision_score(true_labels, student_preds, average='macro') * 100
student_recall = recall_score(true_labels, student_preds, average='macro') * 100

# Per-class AUC-ROC
from sklearn.preprocessing import label_binarize
y_test_bin = label_binarize(true_labels, classes=range(7))
student_auc_per_class = []

for i in range(7):
    try:
        auc = roc_auc_score(y_test_bin[:, i], student_probs[:, i])
        student_auc_per_class.append(auc * 100)
    except:
        student_auc_per_class.append(0.0)

student_auc_macro = np.mean(student_auc_per_class)

print("\n📊 STUDENT MODEL METRICS:")
print("=" * 60)
print(f"Accuracy:           {student_accuracy:.2f}%")
print(f"Macro-Avg F1:       {student_f1_macro:.2f}%")
print(f"Macro-Avg Precision: {student_precision:.2f}%")
print(f"Macro-Avg Recall:    {student_recall:.2f}%")
print(f"Macro-Avg AUC-ROC:   {student_auc_macro:.2f}%")
print("\nPer-Class AUC-ROC:")
for i, name in enumerate(class_full_names):
    print(f"  {name:35s}: {student_auc_per_class[i]:.2f}%")

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(true_labels, student_preds)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - Student Model\n(HAM10000 Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("✅ Confusion matrix plotted!")

In [ ]:
# Detailed classification report
report = classification_report(
    true_labels,
    student_preds,
    target_names=class_full_names,
    digits=4
)

print("\n📋 CLASSIFICATION REPORT (Student Model):")
print("=" * 80)
print(report)

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping"""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate_cam(self, input_image, target_class=None):
        """Generate CAM heatmap"""
        self.model.eval()

        # Forward pass
        output = self.model(input_image)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()

        # Generate CAM
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)

        # Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam, target_class

print("✅ Grad-CAM class defined!")

In [ ]:
# Get the last convolutional layer of student model
# For ConvNeXt, it's in the backbone
target_layer = student_model.backbone.stages[-1].blocks[-1]

# Create Grad-CAM object
gradcam = GradCAM(student_model, target_layer)

# Get some test samples
test_images, test_labels = next(iter(test_loader))

# Select 6 samples
num_samples = 6
sample_indices = np.random.choice(len(test_images), num_samples, replace=False)

fig, axes = plt.subplots(2, num_samples, figsize=(18, 6))

for idx, sample_idx in enumerate(sample_indices):
    # Get image
    img_tensor = test_images[sample_idx:sample_idx+1].to(device)
    true_label = test_labels[sample_idx].item()

    # Generate CAM
    cam, pred_class = gradcam.generate_cam(img_tensor)

    # Denormalize image
    img_display = denormalize(test_images[sample_idx]).permute(1, 2, 0).numpy()
    img_display = np.clip(img_display, 0, 1)

    # Resize CAM to image size
    cam_resized = torch.nn.functional.interpolate(
        cam,
        size=(224, 224),
        mode='bilinear',
        align_corners=False
    )
    cam_np = cam_resized.squeeze().cpu().numpy()

    # Plot original image
    axes[0, idx].imshow(img_display)
    axes[0, idx].set_title(f'True: {lesion_type_dict[idx_to_label[true_label]]}', fontsize=9)
    axes[0, idx].axis('off')

    # Plot Grad-CAM overlay
    axes[1, idx].imshow(img_display)
    axes[1, idx].imshow(cam_np, cmap='jet', alpha=0.5)
    pred_name = lesion_type_dict[idx_to_label[pred_class]]
    axes[1, idx].set_title(f'Pred: {pred_name}', fontsize=9)
    axes[1, idx].axis('off')

plt.suptitle('Grad-CAM Visualization on Student Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Grad-CAM visualizations generated!")

In [ ]:
# Save metrics to text file
with open('evaluation_results.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("SKIN CANCER CLASSIFICATION - EVALUATION RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("STUDENT MODEL METRICS:\n")
    f.write("-" * 80 + "\n")
    f.write(f"Accuracy:            {student_accuracy:.2f}%\n")
    f.write(f"Macro-Avg F1:        {student_f1_macro:.2f}%\n")
    f.write(f"Macro-Avg Precision: {student_precision:.2f}%\n")
    f.write(f"Macro-Avg Recall:    {student_recall:.2f}%\n")
    f.write(f"Macro-Avg AUC-ROC:   {student_auc_macro:.2f}%\n\n")

    f.write("PER-CLASS AUC-ROC:\n")
    f.write("-" * 80 + "\n")
    for i, name in enumerate(class_full_names):
        f.write(f"{name:35s}: {student_auc_per_class[i]:.2f}%\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("CLASSIFICATION REPORT:\n")
    f.write("=" * 80 + "\n")
    f.write(report)

print("✅ Results saved to 'evaluation_results.txt'")

print("\n✅ PHASE 9 COMPLETE!")
print("Evaluation & Grad-CAM done. Tell me: 'Phase 9 done, move to Phase 10'")

# ***PHASE 10: Inference Demo & ONNX Export***

In [ ]:
def predict_single_image(model, image_path, transform, device):
    """
    Predict skin lesion class for a single image

    Args:
        model: Trained model
        image_path: Path to image file
        transform: Albumentations transform
        device: torch device

    Returns:
        prediction: Class name (full)
        confidence: Confidence score
        all_probs: All class probabilities
    """
    model.eval()

    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Apply transform
    augmented = transform(image=image)
    img_tensor = augmented['image'].unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, pred_idx = probs.max(1)

    # Map to full label name (CRITICAL REQUIREMENT)
    pred_abbr = idx_to_label[pred_idx.item()]
    pred_full_name = lesion_type_dict[pred_abbr]
    confidence_pct = confidence.item() * 100

    # Get all probabilities
    all_probs = probs.cpu().numpy()[0]

    return pred_full_name, confidence_pct, all_probs

print("✅ Inference function defined!")

In [ ]:
# Pick a random test image
random_idx = np.random.randint(0, len(test_df))
test_image_path = test_df.iloc[random_idx]['image_path']
true_label_abbr = test_df.iloc[random_idx]['dx']
true_label_full = lesion_type_dict[true_label_abbr]

print(f"Testing inference on: {test_image_path}")
print(f"True label: {true_label_full} ({true_label_abbr})")

# Predict
pred_name, confidence, all_probs = predict_single_image(
    student_model,
    test_image_path,
    val_transform,
    device
)

print(f"\n🔍 PREDICTION:")
print(f"   Class: {pred_name}")
print(f"   Confidence: {confidence:.2f}%")

# Display image with prediction
img = cv2.imread(test_image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 5))

# Original image
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f'True: {true_label_full}\nPred: {pred_name} ({confidence:.1f}%)',
          fontsize=12, fontweight='bold')
plt.axis('off')

# Confidence bar chart
plt.subplot(1, 2, 2)
class_names_full = [lesion_type_dict[k] for k in class_names]
colors = ['green' if p == max(all_probs) else 'steelblue' for p in all_probs]
plt.barh(class_names_full, all_probs * 100, color=colors)
plt.xlabel('Confidence (%)', fontsize=11)
plt.title('Class Probabilities', fontsize=12, fontweight='bold')
plt.xlim(0, 100)
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Inference test complete!")

In [ ]:
def inference_demo():
    """Interactive demo for uploading and predicting images"""
    print("=" * 80)
    print("🩺 SKIN LESION CLASSIFICATION DEMO")
    print("=" * 80)
    print("\nUpload a skin lesion image (.jpg, .jpeg, .png)")
    print("The model will predict the lesion type with confidence scores.\n")

    # Upload file
    from google.colab import files
    uploaded = files.upload()

    if len(uploaded) == 0:
        print("❌ No file uploaded!")
        return

    # Get uploaded filename
    filename = list(uploaded.keys())[0]
    print(f"\n✅ Image uploaded: {filename}")

    # Predict
    pred_name, confidence, all_probs = predict_single_image(
        student_model,
        filename,
        val_transform,
        device
    )

    # Load and display image
    img = cv2.imread(filename)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Create visualization
    fig = plt.figure(figsize=(16, 6))

    # Original Image
    ax1 = plt.subplot(1, 3, 1)
    ax1.imshow(img)
    ax1.set_title('Uploaded Image', fontsize=14, fontweight='bold')
    ax1.axis('off')

    # Prediction Result
    ax2 = plt.subplot(1, 3, 2)
    ax2.text(0.5, 0.6, 'PREDICTION', ha='center', fontsize=18,
             fontweight='bold', transform=ax2.transAxes)
    ax2.text(0.5, 0.4, pred_name, ha='center', fontsize=16,
             color='darkgreen', transform=ax2.transAxes, wrap=True)
    ax2.text(0.5, 0.25, f'Confidence: {confidence:.2f}%', ha='center',
             fontsize=14, transform=ax2.transAxes)
    ax2.axis('off')

    # Confidence Bar Chart
    ax3 = plt.subplot(1, 3, 3)
    class_names_full = [lesion_type_dict[k] for k in class_names]
    colors = ['green' if p == max(all_probs) else 'steelblue' for p in all_probs]
    y_pos = np.arange(len(class_names_full))
    ax3.barh(y_pos, all_probs * 100, color=colors)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(class_names_full, fontsize=10)
    ax3.set_xlabel('Confidence (%)', fontsize=12)
    ax3.set_title('All Class Probabilities', fontsize=14, fontweight='bold')
    ax3.set_xlim(0, 100)
    ax3.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print detailed results
    print("\n" + "=" * 80)
    print("📊 DETAILED RESULTS:")
    print("=" * 80)
    print(f"\n🎯 Predicted Class: {pred_name}")
    print(f"✅ Confidence: {confidence:.2f}%\n")
    print("All Class Probabilities:")
    print("-" * 80)
    for name, prob in zip(class_names_full, all_probs):
        bar = "█" * int(prob * 50)
        print(f"{name:35s} | {bar} {prob*100:5.2f}%")
    print("=" * 80)

print("✅ Demo function created!")
print("\nTo use: Run `inference_demo()` to upload and classify an image")

In [ ]:
# Install onnxscript if not already installed
!pip install -q onnxscript

import torch.onnx

# Load best student model
student_model.load_state_dict(torch.load('student_best.pth'))
student_model.eval()

# Dummy input for tracing
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export to ONNX
print("Exporting model to ONNX format...")
torch.onnx.export(
    student_model,
    dummy_input,
    "model_student.onnx",
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print("✅ Model exported to 'model_student.onnx'")

# Verify ONNX model
import onnx
onnx_model = onnx.load("model_student.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX model verified successfully!")

# Get file size
import os
onnx_size = os.path.getsize("model_student.onnx") / (1024 * 1024)
print(f"   ONNX file size: {onnx_size:.2f} MB")

In [ ]:
# Install onnxruntime if not already installed
!pip install -q onnxruntime

import onnxruntime as ort

# Create ONNX inference session
ort_session = ort.InferenceSession("model_student.onnx")

# Test with same dummy input
dummy_np = dummy_input.cpu().numpy()

# Run inference
onnx_outputs = ort_session.run(
    None,
    {"input": dummy_np}
)

# Compare with PyTorch
with torch.no_grad():
    torch_output = student_model(dummy_input).cpu().numpy()

# Check difference
diff = np.abs(onnx_outputs[0] - torch_output).max()
print(f"\n✅ ONNX inference working!")
print(f"   Max difference vs PyTorch: {diff:.6f}")
print(f"   {'✅ PASS' if diff < 1e-4 else '⚠️ WARNING: Large difference'}")

In [ ]:
print("\n" + "=" * 80)
print("🎉 PROJECT COMPLETE - SUMMARY REPORT")
print("=" * 80)

print("\n📁 FILES GENERATED:")
print("-" * 80)
print("✅ teacher_best_224.pth         - Teacher model weights")
print("✅ student_best.pth             - Student model weights")
print("✅ model_student.onnx           - ONNX export for deployment")
print("✅ evaluation_results.txt       - Detailed metrics")

print("\n📊 MODEL PERFORMANCE:")
print("-" * 80)
print(f"Teacher Test Accuracy:  {teacher_test_acc:.2f}%")
print(f"Student Test Accuracy:  {student_test_acc:.2f}%")
print(f"Accuracy Gap:           {teacher_test_acc - student_test_acc:.2f}%")

print("\n💾 MODEL SIZE:")
print("-" * 80)
print(f"Teacher Parameters:     {teacher_params:,} (~{teacher_params*4/1e6:.1f} MB)")
print(f"Student Parameters:     {student_params:,} (~{student_params*4/1e6:.1f} MB)")
print(f"ONNX File Size:         {onnx_size:.2f} MB")
print(f"Compression Ratio:      {teacher_params/student_params:.1f}x smaller")

print("\n🚀 DEPLOYMENT READY:")
print("-" * 80)
print("✅ Student model optimized for real-time inference")
print("✅ ONNX format compatible with TensorFlow, OpenCV, etc.")
print("✅ Label mapping implemented (outputs full disease names)")
print("✅ Grad-CAM visualization for interpretability")

print("\n📝 USAGE:")
print("-" * 80)
print("To classify a new image:")
print("  1. Run: inference_demo()")
print("  2. Upload a skin lesion image")
print("  3. View prediction with confidence scores")

print("\n" + "=" * 80)
print("✅ PHASE 10 COMPLETE - ALL TASKS FINISHED!")
print("=" * 80)